# R6 — Paper-Style Two-Year Initialization + Weekly Incremental Updating

Training-cadence ablation only: universe, R1 primary target (`target_abs_1pct_next_week`), the predeclared authoritative engineered feature set (full A+B+C+D+E+F+G+H+I set, 34 features -- the same representation R3/R4 used as the R1 baseline, per that established precedent), and the NN v1 / NN v2 architectures/hyperparameters are all held fixed and reused verbatim. Only the training cadence changes: R1's cold-restart-every-fold walk-forward retraining becomes R6's paper-style annual initialization (two preceding calendar years) followed by weekly single-epoch incremental updates that persist model and optimizer state through the trading year, with a fresh annual model at each year boundary. See `docs/weekly_sector_rotation_r6_incremental_update_test.md` for the full write-up; this notebook is the executable source of the CSVs it cites. Does not implement R5 or R7.

In [1]:
"""
R6 full pipeline (scratchpad validation run). Once verified, transcribed into
r6_incremental_update_ablation.ipynb and executed there for the committed
artifacts. Paper-style two-year init + weekly incremental update cadence vs.
the R1 cold-restart-every-fold cadence, same feature set / target / architecture.
"""
import json
import time
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()
REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"

SECTOR_ETFS = ["VGT", "VHT", "VCR", "VOX", "VFH", "VIS", "VDC", "VPU", "VAW", "VNQ", "VDE"]
RANK_K_VALUES = [2, 3]
BLOCK_SIZE = 8
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260812

# ---------------------------------------------------------------------------

## 1. Load panel + R1 primary target, build the predeclared authoritative feature set (full engineered set, 34 features -- precedent from R3/R4), R1/R6 year lists, and the temporal-alignment (leakage) assertion covering every consecutive panel-date pair.

In [2]:
# 1. Load panel + R1 target, build the predeclared authoritative feature set.
# ---------------------------------------------------------------------------
date_cols = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
             "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)
AUDIT_COLUMNS = ["week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
                  "next_week_first_trading_date", "next_week_last_trading_date"]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

r1_targets = pd.read_csv(OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv", parse_dates=["date"])
panel = panel.merge(
    r1_targets[["date", "symbol", "target_abs_1pct_next_week", "next_week_open_to_close_return",
                "next_week_vti_return"]],
    on=["date", "symbol"], how="left",
)

FEATURE_GROUPS = {
    "A": ["abs_return_1w", "abs_return_2w", "abs_return_4w", "abs_return_8w", "abs_return_12w", "abs_return_26w"],
    "B": ["excess_return_vs_vti_1w", "excess_return_vs_vti_2w", "excess_return_vs_vti_4w",
          "excess_return_vs_vti_8w", "excess_return_vs_vti_12w", "excess_return_vs_vti_26w"],
    "C": ["cross_sectional_rank_1w", "cross_sectional_rank_4w", "cross_sectional_rank_8w", "cross_sectional_rank_12w"],
    "D": ["realized_vol_4w", "realized_vol_12w", "realized_vol_26w"],
    "E": ["trend_dist_from_ma_4w", "trend_dist_from_ma_12w", "trend_dist_from_ma_26w"],
    "F": ["drawdown_dist_from_high_13w", "drawdown_dist_from_high_26w"],
    "G": ["volume_change_1w", "volume_vs_avg_4w", "volume_vs_avg_12w"],
    "H": ["vti_return_1w", "vti_return_4w", "vti_return_12w", "vti_volatility_4w"],
    "I": ["sector_return_dispersion_1w", "average_sector_return_1w", "sectors_outperforming_vti_1w"],
}
FULL_FEATURE_COLS = [c for g in "ABCDEFGHI" for c in FEATURE_GROUPS[g]]
assert len(FULL_FEATURE_COLS) == 34
assert set(FULL_FEATURE_COLS).issubset(set(FEATURE_COLUMNS))

md = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"]).copy()
md["label_binary"] = md["target_abs_1pct_next_week"].astype(int)
md["year"] = md["date"].dt.year
years_all = sorted(md["year"].unique())
assert years_all == list(range(2005, 2027))
R1_TEST_YEARS = years_all[3:]
assert R1_TEST_YEARS == list(range(2008, 2027))
R6_TRADING_YEARS = years_all[2:]
assert R6_TRADING_YEARS == list(range(2007, 2027))
COMMON_YEARS = [y for y in R6_TRADING_YEARS if y in R1_TEST_YEARS]
assert COMMON_YEARS == list(range(2008, 2027))
print(f"[{time.time()-t0:.0f}s] data loaded, {len(md)} rows. R6 trading years "
      f"{R6_TRADING_YEARS[0]}-{R6_TRADING_YEARS[-1]} ({len(R6_TRADING_YEARS)}); "
      f"common-with-R1 years {COMMON_YEARS[0]}-{COMMON_YEARS[-1]} ({len(COMMON_YEARS)})")

ALL_DATES = sorted(md["date"].unique())
DATE_LABEL_KNOWN = md.groupby("date")["next_week_last_trading_date"].max()
DATE_FEATURE_CUTOFF = md.groupby("date")["feature_cutoff_timestamp"].min()
for d_prev, d_next in zip(ALL_DATES[:-1], ALL_DATES[1:]):
    assert DATE_LABEL_KNOWN.loc[d_prev] < DATE_FEATURE_CUTOFF.loc[d_next], \
        f"temporal leak risk between {d_prev} and {d_next}"
print(f"[{time.time()-t0:.0f}s] temporal alignment assertion passed for {len(ALL_DATES)-1} consecutive date pairs")

# ---------------------------------------------------------------------------

[0s] data loaded, 12253 rows. R6 trading years 2007-2026 (20); common-with-R1 years 2008-2026 (19)
[0s] temporal alignment assertion passed for 1113 consecutive date pairs


## 2. Model definitions -- verbatim R1 `SmallMLP` (NN v1) / `WideSingleLayerMLP` seed-ensemble (NN v2) hidden config and hyperparameters -- plus the stateful-year runner: two-year init (fit/val early-stopped fit, same rule as R1), fixed annual scaler, and the weekly incremental-update loop (1 predeclared epoch per newly-known week, persisted model + optimizer state across the year).

In [3]:
# 2. Model definitions -- verbatim R1 hidden config / hyperparameters.
# ---------------------------------------------------------------------------
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def init_fit_mlp(X_fit, y_fit, X_val, y_val, n_features, seed=NN_SEED):
    torch.manual_seed(seed)
    model = SmallMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(MAX_EPOCHS):
        model.train(); optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                break
    model.load_state_dict(best_state); model.eval()
    return model


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def init_fit_mlp_v2(X_fit, y_fit, X_val, y_val, n_features, seed):
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=V2_LEARNING_RATE, weight_decay=V2_WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(V2_MAX_EPOCHS):
        model.train(); optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= V2_PATIENCE:
                break
    model.load_state_dict(best_state); model.eval()
    return model


def flat_params(model):
    return torch.cat([p.detach().flatten() for p in model.parameters()])


def one_incremental_epoch(model, optimizer, X, y, loss_fn):
    model.train()
    optimizer.zero_grad()
    X_t = torch.tensor(X, dtype=torch.float32); y_t = torch.tensor(y, dtype=torch.float32)
    loss = loss_fn(model(X_t), y_t)
    loss.backward(); optimizer.step()
    model.eval()
    return float(loss.item())


def run_stateful_year(md, year, feature_cols, v2, diag_store, forgetting_store, update_loss_store):
    fit_year, val_year = year - 2, year - 1
    fit_mask = md["year"] == fit_year
    val_mask = md["year"] == val_year
    init_mask = fit_mask | val_mask
    year_mask = md["year"] == year
    scaler = StandardScaler().fit(md.loc[init_mask, feature_cols])
    X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
    X_val = scaler.transform(md.loc[val_mask, feature_cols])
    y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
    y_val = md.loc[val_mask, "label_binary"].to_numpy()
    n_features = len(feature_cols)
    loss_fn = nn.BCEWithLogitsLoss()

    year_rows = md.loc[year_mask].copy()
    dates_year = sorted(year_rows["date"].unique())

    n_models = N_SEEDS if v2 else 1
    models, optimizers, init_param_vecs = [], [], []
    forget_auc_init, forget_loss_init = [], []
    for si in range(n_models):
        seed = SEED_BASE + si if v2 else NN_SEED
        model = init_fit_mlp_v2(X_fit, y_fit, X_val, y_val, n_features, seed) if v2 else \
            init_fit_mlp(X_fit, y_fit, X_val, y_val, n_features, seed)
        optimizer = torch.optim.Adam(model.parameters(),
                                      lr=(V2_LEARNING_RATE if v2 else LEARNING_RATE),
                                      weight_decay=(V2_WEIGHT_DECAY if v2 else WEIGHT_DECAY))
        models.append(model); optimizers.append(optimizer)
        init_param_vecs.append(flat_params(model).clone())
        with torch.no_grad():
            p_val = torch.sigmoid(model(torch.tensor(X_val, dtype=torch.float32))).numpy()
        forget_auc_init.append(roc_auc_score(y_val, p_val) if len(np.unique(y_val)) > 1 else np.nan)
        forget_loss_init.append(float(nn.functional.binary_cross_entropy(
            torch.tensor(p_val), torch.tensor(y_val, dtype=torch.float32)).item()))

    diag_dates = dates_year[:1]
    X_diag = scaler.transform(year_rows[year_rows["date"].isin(diag_dates)][feature_cols])
    with torch.no_grad():
        preds_diag_start = [torch.sigmoid(models[si](torch.tensor(X_diag, dtype=torch.float32))).numpy()
                             for si in range(n_models)]

    oof_rows = []
    n_updates = 0
    for idx, d in enumerate(dates_year):
        if idx > 0:
            d_prev = dates_year[idx - 1]
            prev_rows = year_rows[year_rows["date"] == d_prev]
            X_prev = scaler.transform(prev_rows[feature_cols])
            y_prev = prev_rows["label_binary"].to_numpy()
            for si in range(n_models):
                loss_val = one_incremental_epoch(models[si], optimizers[si], X_prev, y_prev, loss_fn)
                update_loss_store.append({"year": year, "v2": v2, "seed_idx": si, "update_index": idx,
                                           "date_updated": d_prev, "n_rows": len(prev_rows), "train_loss": loss_val})
            n_updates += 1

        rows_d = year_rows[year_rows["date"] == d]
        X_d = scaler.transform(rows_d[feature_cols])
        with torch.no_grad():
            seed_preds = [torch.sigmoid(models[si](torch.tensor(X_d, dtype=torch.float32))).numpy()
                          for si in range(n_models)]
        pred = np.mean(seed_preds, axis=0)
        out = rows_d[["date", "symbol", "next_week_open_to_close_return", "next_week_vti_return",
                       "label_binary"]].copy()
        out["predicted_proba"] = pred
        oof_rows.append(out)

    oof = pd.concat(oof_rows, ignore_index=True)

    with torch.no_grad():
        preds_diag_end = [torch.sigmoid(models[si](torch.tensor(X_diag, dtype=torch.float32))).numpy()
                           for si in range(n_models)]
        p_val_end = [torch.sigmoid(models[si](torch.tensor(X_val, dtype=torch.float32))).numpy()
                     for si in range(n_models)]

    for si in range(n_models):
        end_param_vec = flat_params(models[si])
        weight_norm_change = float(torch.norm(end_param_vec - init_param_vecs[si]).item())
        mean_abs_pred_drift = float(np.mean(np.abs(preds_diag_end[si] - preds_diag_start[si])))
        auc_end = roc_auc_score(y_val, p_val_end[si]) if len(np.unique(y_val)) > 1 else np.nan
        loss_end = float(nn.functional.binary_cross_entropy(
            torch.tensor(p_val_end[si]), torch.tensor(y_val, dtype=torch.float32)).item())
        diag_store.append({
            "year": year, "v2": v2, "seed_idx": si, "n_weekly_updates": n_updates,
            "weight_norm_change_from_init": weight_norm_change,
            "mean_abs_prediction_drift_frozen_sample": mean_abs_pred_drift,
            "n_diag_sample_rows": len(X_diag),
        })
        forgetting_store.append({
            "year": year, "v2": v2, "seed_idx": si, "val_year": val_year, "n_val_rows": len(X_val),
            "auc_init": forget_auc_init[si], "auc_end_of_year": auc_end,
            "auc_drop": (forget_auc_init[si] - auc_end) if not np.isnan(forget_auc_init[si]) and not np.isnan(auc_end) else np.nan,
            "loss_init": forget_loss_init[si], "loss_end_of_year": loss_end,
            "loss_increase": loss_end - forget_loss_init[si],
        })

    fold_row = {
        "year": year, "fit_year": fit_year, "val_year": val_year,
        "n_init_train": int(fit_mask.sum()), "n_init_val": int(val_mask.sum()),
        "n_year_rows": len(oof), "n_weekly_updates": n_updates, "n_weeks_in_year": len(dates_year),
        "init_train_positive_rate": float(y_fit.mean()), "init_val_positive_rate": float(y_val.mean()),
        "year_positive_rate": float(oof["label_binary"].mean()),
    }
    y_oof = oof["label_binary"].to_numpy(); s_oof = oof["predicted_proba"].to_numpy()
    fold_row["year_auc"] = roc_auc_score(y_oof, s_oof) if len(np.unique(y_oof)) > 1 else np.nan
    fold_row["year_pr_auc"] = average_precision_score(y_oof, s_oof) if len(np.unique(y_oof)) > 1 else np.nan
    fold_row["year_brier"] = brier_score_loss(y_oof, s_oof)

    return oof, fold_row


print(f"[{time.time()-t0:.0f}s] Step 2 (model defs + stateful-year runner) done.")

# ---------------------------------------------------------------------------

[0s] Step 2 (model defs + stateful-year runner) done.


## 3. Run R6 for NN v1 and NN v2 across all 20 trading years (2007-2026).

In [4]:
# 3. Run R6 for NN v1 and NN v2, all 20 trading years (2007-2026).
# ---------------------------------------------------------------------------
diag_store, forget_store, update_loss_store = [], [], []
fold_rows_v1, fold_rows_v2 = [], []
oof_v1_by_year, oof_v2_by_year = {}, {}

for year in R6_TRADING_YEARS:
    oof, fold = run_stateful_year(md, year, FULL_FEATURE_COLS, v2=False,
                                   diag_store=diag_store, forgetting_store=forget_store,
                                   update_loss_store=update_loss_store)
    fold_rows_v1.append(fold); oof_v1_by_year[year] = oof
    print(f"[{time.time()-t0:.0f}s] R6 NN_v1 year={year}: auc={fold['year_auc']:.4f} n_updates={fold['n_weekly_updates']}")

for year in R6_TRADING_YEARS:
    oof, fold = run_stateful_year(md, year, FULL_FEATURE_COLS, v2=True,
                                   diag_store=diag_store, forgetting_store=forget_store,
                                   update_loss_store=update_loss_store)
    fold_rows_v2.append(fold); oof_v2_by_year[year] = oof
    print(f"[{time.time()-t0:.0f}s] R6 NN_v2 year={year}: auc={fold['year_auc']:.4f} n_updates={fold['n_weekly_updates']}")

r6_fold_metrics = pd.concat([
    pd.DataFrame(fold_rows_v1).assign(model="NN_v1"),
    pd.DataFrame(fold_rows_v2).assign(model="NN_v2"),
], ignore_index=True)

oof_v1_all = pd.concat(oof_v1_by_year.values(), ignore_index=True)
oof_v2_all = pd.concat(oof_v2_by_year.values(), ignore_index=True)
print(f"[{time.time()-t0:.0f}s] Step 3 (R6 stateful runs, both models, all years) done.")

# ---------------------------------------------------------------------------

[1s] R6 NN_v1 year=2007: auc=0.6084 n_updates=51
[1s] R6 NN_v1 year=2008: auc=0.5792 n_updates=51


[1s] R6 NN_v1 year=2009: auc=0.4795 n_updates=51
[1s] R6 NN_v1 year=2010: auc=0.5169 n_updates=52


[1s] R6 NN_v1 year=2011: auc=0.5724 n_updates=51
[2s] R6 NN_v1 year=2012: auc=0.6374 n_updates=51


[2s] R6 NN_v1 year=2013: auc=0.5696 n_updates=51
[2s] R6 NN_v1 year=2014: auc=0.5601 n_updates=51


[2s] R6 NN_v1 year=2015: auc=0.5699 n_updates=51
[2s] R6 NN_v1 year=2016: auc=0.5696 n_updates=52


...

[9s] R6 NN_v2 year=2022: auc=0.5380 n_updates=51


[9s] R6 NN_v2 year=2023: auc=0.4710 n_updates=51


[10s] R6 NN_v2 year=2024: auc=0.4566 n_updates=51


[10s] R6 NN_v2 year=2025: auc=0.5107 n_updates=51


[10s] R6 NN_v2 year=2026: auc=0.5057 n_updates=30
[10s] Step 3 (R6 stateful runs, both models, all years) done.


## 4. Top-K metrics and block-bootstrap utilities -- verbatim reuse of R1/R3/R4's implementations.

In [5]:
# 4. Top-K metrics + block bootstrap (verbatim R1/R3/R4 methodology).
# ---------------------------------------------------------------------------
def topk_metrics_and_series(oof: pd.DataFrame, base_rate: float, k: int):
    weekly_precision, weekly_any_pos, weekly_all_pos = [], [], []
    weekly_mean_ret, weekly_all_sector_mean, weekly_vti_ret, weekly_bottomk_mean = [], [], [], []
    dates_used = []
    for date, g in oof.groupby("date"):
        if len(g) < 11:
            continue
        ranked = g["predicted_proba"].rank(ascending=False, method="first")
        n = len(g)
        top = g.loc[ranked <= k]; bottom = g.loc[ranked > n - k]
        if top["next_week_open_to_close_return"].isna().any() or top["label_binary"].isna().any():
            continue
        weekly_precision.append(top["label_binary"].mean())
        weekly_any_pos.append(float((top["label_binary"] == 1).any()))
        weekly_all_pos.append(float((top["label_binary"] == 1).all()))
        weekly_mean_ret.append(top["next_week_open_to_close_return"].mean())
        weekly_all_sector_mean.append(g["next_week_open_to_close_return"].mean())
        weekly_vti_ret.append(g["next_week_vti_return"].iloc[0])
        weekly_bottomk_mean.append(bottom["next_week_open_to_close_return"].mean())
        dates_used.append(date)
    series = {
        "date": dates_used, "precision": pd.Series(weekly_precision), "selected_ret": pd.Series(weekly_mean_ret),
        "all_sector_mean": pd.Series(weekly_all_sector_mean), "vti_ret": pd.Series(weekly_vti_ret),
        "bottomk_mean": pd.Series(weekly_bottomk_mean),
    }
    row = {
        "k": k, "n_weeks": len(dates_used), "precision_at_k": series["precision"].mean(),
        "base_positive_rate": base_rate, "precision_minus_base": series["precision"].mean() - base_rate,
        "pct_weeks_at_least_one_positive": pd.Series(weekly_any_pos).mean(),
        "pct_weeks_all_positive": pd.Series(weekly_all_pos).mean(),
        "mean_selected_return": series["selected_ret"].mean(), "median_selected_return": series["selected_ret"].median(),
        "selected_minus_all_sector_mean": (series["selected_ret"] - series["all_sector_mean"]).mean(),
        "selected_minus_vti": (series["selected_ret"] - series["vti_ret"]).mean(),
        "topk_minus_bottomk_spread": (series["selected_ret"] - series["bottomk_mean"]).mean(),
        "win_rate_vs_vti": (series["selected_ret"] > series["vti_ret"]).mean(),
    }
    return row, series


def block_bootstrap_ci(values: pd.Series, block_size: int = BLOCK_SIZE, n_boot: int = N_BOOTSTRAP,
                        seed: int = BOOTSTRAP_SEED) -> dict:
    arr = values.to_numpy(); n = len(arr)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size)); max_start = n - block_size
    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        sample = np.concatenate([arr[s: s + block_size] for s in starts])[:n]
        boot_means[b] = sample.mean()
    lower, upper = np.percentile(boot_means, [2.5, 97.5])
    return {"point_estimate": arr.mean(), "ci_lower": lower, "ci_upper": upper,
            "n_weeks": n, "significant": bool(lower > 0 or upper < 0)}


def _weekly_yscore_lists(oof: pd.DataFrame):
    dates = sorted(oof["date"].unique())
    y_list, s_list = [], []
    for d in dates:
        g = oof[oof["date"] == d]
        y_list.append(g["label_binary"].to_numpy()); s_list.append(g["predicted_proba"].to_numpy())
    return dates, y_list, s_list


def block_bootstrap_auc_edge(oof: pd.DataFrame, block_size: int = BLOCK_SIZE, n_boot: int = N_BOOTSTRAP,
                              seed: int = BOOTSTRAP_SEED) -> dict:
    dates, y_list, s_list = _weekly_yscore_lists(oof)
    n = len(dates); rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size)); max_start = n - block_size
    point_auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])
    boot_aucs = np.full(n_boot, np.nan)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        idxs = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
        y = np.concatenate([y_list[i] for i in idxs]); s = np.concatenate([s_list[i] for i in idxs])
        if len(np.unique(y)) > 1:
            boot_aucs[b] = roc_auc_score(y, s)
    valid = boot_aucs[~np.isnan(boot_aucs)]
    lower, upper = np.percentile(valid, [2.5, 97.5])
    return {"point_estimate": point_auc - 0.5, "ci_lower": lower - 0.5, "ci_upper": upper - 0.5,
            "n_weeks": n, "significant": bool((lower - 0.5) > 0 or (upper - 0.5) < 0)}


def paired_block_bootstrap_auc_diff(oof_a: pd.DataFrame, oof_b: pd.DataFrame, block_size: int = BLOCK_SIZE,
                                     n_boot: int = N_BOOTSTRAP, seed: int = BOOTSTRAP_SEED) -> dict:
    common_dates = sorted(set(oof_a["date"].unique()) & set(oof_b["date"].unique()))
    oof_a = oof_a[oof_a["date"].isin(common_dates)]; oof_b = oof_b[oof_b["date"].isin(common_dates)]
    dates_a, y_list_a, s_list_a = _weekly_yscore_lists(oof_a)
    dates_b, y_list_b, s_list_b = _weekly_yscore_lists(oof_b)
    assert dates_a == dates_b
    n = len(dates_a); rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size)); max_start = n - block_size
    point_auc_a = roc_auc_score(oof_a["label_binary"], oof_a["predicted_proba"])
    point_auc_b = roc_auc_score(oof_b["label_binary"], oof_b["predicted_proba"])
    boot_diffs = np.full(n_boot, np.nan)
    for it in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        idxs = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
        y_a = np.concatenate([y_list_a[i] for i in idxs]); s_a = np.concatenate([s_list_a[i] for i in idxs])
        y_b = np.concatenate([y_list_b[i] for i in idxs]); s_b = np.concatenate([s_list_b[i] for i in idxs])
        if len(np.unique(y_a)) > 1 and len(np.unique(y_b)) > 1:
            boot_diffs[it] = roc_auc_score(y_a, s_a) - roc_auc_score(y_b, s_b)
    valid = boot_diffs[~np.isnan(boot_diffs)]
    lower, upper = np.percentile(valid, [2.5, 97.5])
    point = point_auc_a - point_auc_b
    return {"point_estimate": point, "ci_lower": lower, "ci_upper": upper, "n_weeks": n,
            "significant": bool(lower > 0 or upper < 0)}


def paired_series_diff_bootstrap(oof_a, oof_b, base_rate_a, base_rate_b, k):
    _, series_a = topk_metrics_and_series(oof_a, base_rate_a, k)
    _, series_b = topk_metrics_and_series(oof_b, base_rate_b, k)
    dfa = pd.DataFrame({"date": series_a["date"], "precision": series_a["precision"],
                         "sel_minus_all": series_a["selected_ret"] - series_a["all_sector_mean"],
                         "spread": series_a["selected_ret"] - series_a["bottomk_mean"],
                         "vti_edge": series_a["selected_ret"] - series_a["vti_ret"]}).set_index("date")
    dfb = pd.DataFrame({"date": series_b["date"], "precision": series_b["precision"],
                         "sel_minus_all": series_b["selected_ret"] - series_b["all_sector_mean"],
                         "spread": series_b["selected_ret"] - series_b["bottomk_mean"],
                         "vti_edge": series_b["selected_ret"] - series_b["vti_ret"]}).set_index("date")
    common = dfa.join(dfb, lsuffix="_a", rsuffix="_b", how="inner")
    out = {}
    for qname, col in [("precision_minus_base_diff", "precision"), ("topk_minus_allsector_diff", "sel_minus_all"),
                        ("topk_minus_bottomk_diff", "spread"), ("topk_minus_vti_diff", "vti_edge")]:
        diff_series = common[f"{col}_a"] - common[f"{col}_b"]
        out[qname] = block_bootstrap_ci(diff_series)
    return out


print(f"[{time.time()-t0:.0f}s] Step 4 (topk + bootstrap utilities) defined.")

# ---------------------------------------------------------------------------

[10s] Step 4 (topk + bootstrap utilities) defined.


## 5. Aggregate, stability, and Top-K metrics for R6 (full R6 period 2007-2026, and the common-with-R1 period 2008-2026).

In [6]:
# 5. Aggregate / stability / Top-K metrics for R6 (full period + common period).
# ---------------------------------------------------------------------------
aggregate_rows = []
topk_rows = []
stability_rows = []
for model_name, oof_all, fold_df in [("NN_v1", oof_v1_all, r6_fold_metrics[r6_fold_metrics.model == "NN_v1"]),
                                       ("NN_v2", oof_v2_all, r6_fold_metrics[r6_fold_metrics.model == "NN_v2"])]:
    for period_name, years_subset in [("full_r6_period", R6_TRADING_YEARS), ("common_with_r1", COMMON_YEARS)]:
        oof_p = oof_all[oof_all["date"].dt.year.isin(years_subset)]
        fold_p = fold_df[fold_df["year"].isin(years_subset)]
        auc = roc_auc_score(oof_p["label_binary"], oof_p["predicted_proba"])
        prauc = average_precision_score(oof_p["label_binary"], oof_p["predicted_proba"])
        brier = brier_score_loss(oof_p["label_binary"], oof_p["predicted_proba"])
        base_rate = oof_p["label_binary"].mean()
        aggregate_rows.append({
            "model": model_name, "period": period_name, "n_years": len(years_subset), "n_oof_rows": len(oof_p),
            "pooled_auc": auc, "pooled_pr_auc": prauc, "pooled_brier": brier,
            "mean_year_auc": fold_p["year_auc"].mean(), "median_year_auc": fold_p["year_auc"].median(),
            "std_year_auc": fold_p["year_auc"].std(), "worst_year_auc": fold_p["year_auc"].min(),
            "best_year_auc": fold_p["year_auc"].max(), "base_positive_rate": base_rate,
        })
        stability_rows.append({
            "model": model_name, "period": period_name,
            "std_year_auc": fold_p["year_auc"].std(), "worst_year_auc": fold_p["year_auc"].min(),
            "best_year_auc": fold_p["year_auc"].max(),
            "frac_years_auc_above_0.5": float((fold_p["year_auc"] > 0.5).mean()),
        })
        for k in RANK_K_VALUES:
            row, _ = topk_metrics_and_series(oof_p, base_rate, k)
            row.update({"model": model_name, "period": period_name})
            topk_rows.append(row)

r6_aggregate_metrics = pd.DataFrame(aggregate_rows)
r6_stability = pd.DataFrame(stability_rows)
r6_topk_all = pd.DataFrame(topk_rows)
print(f"[{time.time()-t0:.0f}s] Step 5 (aggregate/stability/topk) done.")
print(r6_aggregate_metrics[["model", "period", "pooled_auc", "mean_year_auc", "std_year_auc"]].to_string())

# ---------------------------------------------------------------------------

[13s] Step 5 (aggregate/stability/topk) done.
   model          period  pooled_auc  mean_year_auc  std_year_auc
0  NN_v1  full_r6_period    0.550842       0.554239      0.047510
1  NN_v1  common_with_r1    0.549099       0.551389      0.047022
2  NN_v2  full_r6_period    0.552836       0.553999      0.061383
3  NN_v2  common_with_r1    0.550748       0.551117      0.061659


## 6. Standalone block-bootstrap CIs (R6 vs. chance).

In [7]:
# 6. Standalone bootstrap CIs (R6 vs chance), common period.
# ---------------------------------------------------------------------------
bootstrap_rows = []
for model_name, oof_all in [("NN_v1", oof_v1_all), ("NN_v2", oof_v2_all)]:
    for period_name, years_subset in [("full_r6_period", R6_TRADING_YEARS), ("common_with_r1", COMMON_YEARS)]:
        oof_p = oof_all[oof_all["date"].dt.year.isin(years_subset)]
        base_rate = oof_p["label_binary"].mean()
        b = block_bootstrap_auc_edge(oof_p)
        bootstrap_rows.append({"model": model_name, "period": period_name, "k": np.nan,
                                "quantity": "auc_edge_over_0.5", **b})
        for k in RANK_K_VALUES:
            _, series = topk_metrics_and_series(oof_p, base_rate, k)
            b = block_bootstrap_ci(series["precision"])
            bootstrap_rows.append({"model": model_name, "period": period_name, "k": k,
                                    "quantity": "precision_minus_base",
                                    "point_estimate": b["point_estimate"] - base_rate,
                                    "ci_lower": b["ci_lower"] - base_rate, "ci_upper": b["ci_upper"] - base_rate,
                                    "n_weeks": b["n_weeks"], "significant": b["significant"]})
            for qname, diff in [
                ("selected_minus_all_sector_mean", series["selected_ret"] - series["all_sector_mean"]),
                ("selected_minus_vti", series["selected_ret"] - series["vti_ret"]),
                ("topk_minus_bottomk_spread", series["selected_ret"] - series["bottomk_mean"]),
            ]:
                b = block_bootstrap_ci(diff)
                bootstrap_rows.append({"model": model_name, "period": period_name, "k": k, "quantity": qname, **b})
r6_bootstrap_ci = pd.DataFrame(bootstrap_rows)
print(f"[{time.time()-t0:.0f}s] Step 6 (standalone bootstrap) done.")

# ---------------------------------------------------------------------------

[65s] Step 6 (standalone bootstrap) done.


## 7. Load R1's saved full-feature-set OOF (verbatim cold-restart-every-fold cadence, already validated in R3/R4) and run the paired R6-vs-R1 block bootstrap on the common 2008-2026 weeks -- the primary R6 statistical test.

In [8]:
# 7. R1 baseline OOF (full-feature-set, cold-restart cadence) -- reload from
#    R3's saved regeneration (verbatim R1 code, already sanity-checked in R4).
# ---------------------------------------------------------------------------
r1_baseline_oof = {}
for name in ["NN_v1", "NN_v2"]:
    r1_baseline_oof[name] = pd.read_csv(OUTPUTS_ROOT / f"r3_r1_baseline_oof_{name}.csv", parse_dates=["date"])
    r1_baseline_oof[name] = r1_baseline_oof[name][r1_baseline_oof[name]["date"].dt.year.isin(COMMON_YEARS)]

paired_rows = []
for model_name, oof_all in [("NN_v1", oof_v1_all), ("NN_v2", oof_v2_all)]:
    oof_r6 = oof_all[oof_all["date"].dt.year.isin(COMMON_YEARS)]
    oof_r1 = r1_baseline_oof[model_name]
    base_r6 = oof_r6["label_binary"].mean(); base_r1 = oof_r1["label_binary"].mean()

    auc_diff = paired_block_bootstrap_auc_diff(oof_r6, oof_r1)
    paired_rows.append({"model": model_name, "k": np.nan, "quantity": "auc", **auc_diff})
    for k in RANK_K_VALUES:
        diffs = paired_series_diff_bootstrap(oof_r6, oof_r1, base_r6, base_r1, k)
        for qname, b in diffs.items():
            paired_rows.append({"model": model_name, "k": k, "quantity": qname, **b})
    print(f"[{time.time()-t0:.0f}s] {model_name} paired R6-vs-R1 (common {COMMON_YEARS[0]}-{COMMON_YEARS[-1]}): "
          f"auc_diff={auc_diff['point_estimate']:.4f} (sig={auc_diff['significant']})")

r6_vs_r1_paired = pd.DataFrame(paired_rows)
print(f"[{time.time()-t0:.0f}s] Step 7 (paired R6-vs-R1 bootstrap) done.")

# ---------------------------------------------------------------------------

[90s] NN_v1 paired R6-vs-R1 (common 2008-2026): auc_diff=-0.0203 (sig=True)


[114s] NN_v2 paired R6-vs-R1 (common 2008-2026): auc_diff=-0.0201 (sig=False)
[114s] Step 7 (paired R6-vs-R1 bootstrap) done.


## 8. Catastrophic-forgetting classification: for each (model, year, seed), the frozen initialization-validation slice's AUC/loss is measured immediately after initialization and again after the year's full sequence of weekly updates (never trained on after initialization).

In [9]:
# 8. Forgetting classification + summary.
# ---------------------------------------------------------------------------
r6_forgetting = pd.DataFrame(forget_store)
r6_forgetting["model"] = np.where(r6_forgetting["v2"], "NN_v2", "NN_v1")


def classify_forgetting(auc_drop):
    if pd.isna(auc_drop):
        return "undefined"
    if auc_drop < 0.01:
        return "negligible"
    if auc_drop < 0.03:
        return "mild"
    if auc_drop < 0.06:
        return "material"
    return "severe"


r6_forgetting["forgetting_class"] = r6_forgetting["auc_drop"].apply(classify_forgetting)
forgetting_summary = r6_forgetting.groupby("model")["forgetting_class"].value_counts().unstack(fill_value=0)
mean_auc_drop = r6_forgetting.groupby("model")["auc_drop"].mean()
print(f"[{time.time()-t0:.0f}s] Step 8 (forgetting classification) done.")
print(forgetting_summary)
print(mean_auc_drop)

r6_update_diagnostics = pd.DataFrame(diag_store)
r6_update_diagnostics["model"] = np.where(r6_update_diagnostics["v2"], "NN_v2", "NN_v1")
r6_update_loss = pd.DataFrame(update_loss_store)

print(f"[{time.time()-t0:.0f}s] ALL DONE (part 1). Saving artifacts next.")

# ---------------------------------------------------------------------------

[114s] Step 8 (forgetting classification) done.
forgetting_class  material  mild  negligible
model                                       
NN_v1                    0     2          18
NN_v2                   12     8          80
model
NN_v1   -0.002384
NN_v2   -0.000986
Name: auc_drop, dtype: float64
[114s] ALL DONE (part 1). Saving artifacts next.


## 9. Save all primary R6 artifacts.

In [10]:
# 9. Save artifacts.
# ---------------------------------------------------------------------------
r6_fold_metrics.to_csv(OUTPUTS_ROOT / "r6_fold_or_year_metrics.csv", index=False)
r6_aggregate_metrics.to_csv(OUTPUTS_ROOT / "r6_aggregate_metrics.csv", index=False)
r6_topk_all[r6_topk_all["k"] == 2].to_csv(OUTPUTS_ROOT / "r6_top2_metrics.csv", index=False)
r6_topk_all[r6_topk_all["k"] == 3].to_csv(OUTPUTS_ROOT / "r6_top3_metrics.csv", index=False)
r6_bootstrap_ci.to_csv(OUTPUTS_ROOT / "r6_bootstrap_ci.csv", index=False)
r6_vs_r1_paired.to_csv(OUTPUTS_ROOT / "r6_vs_r1_paired_comparison.csv", index=False)
r6_update_diagnostics.to_csv(OUTPUTS_ROOT / "r6_update_diagnostics.csv", index=False)
r6_forgetting.to_csv(OUTPUTS_ROOT / "r6_forgetting_diagnostics.csv", index=False)
r6_stability.to_csv(OUTPUTS_ROOT / "r6_stability_metrics.csv", index=False)

sample_audit_rows = []
for model_name, fold_df in [("NN_v1", r6_fold_metrics[r6_fold_metrics.model == "NN_v1"]),
                             ("NN_v2", r6_fold_metrics[r6_fold_metrics.model == "NN_v2"])]:
    for _, r in fold_df.iterrows():
        sample_audit_rows.append({
            "model": model_name, "year": int(r["year"]), "fit_year": int(r["fit_year"]), "val_year": int(r["val_year"]),
            "n_init_train": int(r["n_init_train"]), "n_init_val": int(r["n_init_val"]),
            "n_weeks_in_year": int(r["n_weeks_in_year"]), "n_weekly_updates": int(r["n_weekly_updates"]),
            "n_year_rows": int(r["n_year_rows"]), "init_train_positive_rate": r["init_train_positive_rate"],
            "init_val_positive_rate": r["init_val_positive_rate"], "year_positive_rate": r["year_positive_rate"],
            "in_common_r1_comparison_period": bool(int(r["year"]) in COMMON_YEARS),
        })
r6_sample_audit = pd.DataFrame(sample_audit_rows)
r6_sample_audit.to_csv(OUTPUTS_ROOT / "r6_sample_audit.csv", index=False)

max_auc_drop = float(r6_forgetting["auc_drop"].max())
forgetting_overall = classify_forgetting(max_auc_drop) if not pd.isna(max_auc_drop) else "undefined"
replay_triggered = forgetting_overall in ("material", "severe")

summary = {
    "primary_feature_set": "A+B+C+D+E+F+G+H+I (full engineered set, 34 features) -- precedent from R3/R4",
    "r6_trading_years": [int(R6_TRADING_YEARS[0]), int(R6_TRADING_YEARS[-1])],
    "common_r1_comparison_years": [int(COMMON_YEARS[0]), int(COMMON_YEARS[-1])],
    "nn_v1_r6_pooled_auc_full_period": float(r6_aggregate_metrics[(r6_aggregate_metrics.model == "NN_v1") & (r6_aggregate_metrics.period == "full_r6_period")]["pooled_auc"].iloc[0]),
    "nn_v1_r6_pooled_auc_common_period": float(r6_aggregate_metrics[(r6_aggregate_metrics.model == "NN_v1") & (r6_aggregate_metrics.period == "common_with_r1")]["pooled_auc"].iloc[0]),
    "nn_v2_r6_pooled_auc_full_period": float(r6_aggregate_metrics[(r6_aggregate_metrics.model == "NN_v2") & (r6_aggregate_metrics.period == "full_r6_period")]["pooled_auc"].iloc[0]),
    "nn_v2_r6_pooled_auc_common_period": float(r6_aggregate_metrics[(r6_aggregate_metrics.model == "NN_v2") & (r6_aggregate_metrics.period == "common_with_r1")]["pooled_auc"].iloc[0]),
    "r1_baseline_nn_v1_auc_full_feature_set": float(roc_auc_score(r1_baseline_oof["NN_v1"].label_binary, r1_baseline_oof["NN_v1"].predicted_proba)),
    "r1_baseline_nn_v2_auc_full_feature_set": float(roc_auc_score(r1_baseline_oof["NN_v2"].label_binary, r1_baseline_oof["NN_v2"].predicted_proba)),
    "max_auc_drop_forgetting": max_auc_drop,
    "forgetting_classification_overall": forgetting_overall,
    "replay_sensitivity_triggered": replay_triggered,
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "r6_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# ---------------------------------------------------------------------------

## 10. `R6_REPLAY_SENSITIVITY` -- triggered because NN v2's forgetting diagnostic shows a clear, ensemble-consistent material forgetting pattern concentrated in 2 of 20 years (not per-seed noise), while NN v1 never crosses the material threshold. One predeclared bounded sensitivity for NN v2 only: each weekly update trains on the newly available week's rows UNION a fixed trailing replay buffer of the previous 12 completed training weeks.

In [11]:
# 10. R6_REPLAY_SENSITIVITY -- triggered because NN_v2's forgetting diagnostic
#     (Step 8) shows a clear, ensemble-consistent (not per-seed-noise) material
#     forgetting pattern concentrated in 2 of 20 years (2015, 2016: all 5 seeds
#     cross the material 0.03 AUC-drop threshold together), while NN_v1 never
#     crosses it. One predeclared bounded sensitivity for NN_v2 only: at each
#     weekly update, train on the newly available week's rows UNION a fixed
#     trailing replay buffer of the previous 12 completed training weeks
#     (REPLAY_WINDOW=12, chosen for simplicity per the mandate, not tuned).
# ---------------------------------------------------------------------------
if replay_triggered:
    from collections import deque

    REPLAY_WINDOW = 12

    def run_stateful_year_v2_replay(md, year, feature_cols, diag_store, forgetting_store):
        fit_year, val_year = year - 2, year - 1
        fit_mask = md["year"] == fit_year
        val_mask = md["year"] == val_year
        init_mask = fit_mask | val_mask
        year_mask = md["year"] == year
        scaler = StandardScaler().fit(md.loc[init_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()
        n_features = len(feature_cols)
        loss_fn = nn.BCEWithLogitsLoss()

        year_rows = md.loc[year_mask].copy()
        dates_year = sorted(year_rows["date"].unique())
        init_rows_sorted = md.loc[init_mask].sort_values("date")
        init_dates_sorted = sorted(init_rows_sorted["date"].unique())
        buffer = deque(
            [init_rows_sorted[init_rows_sorted["date"] == d] for d in init_dates_sorted[-REPLAY_WINDOW:]],
            maxlen=REPLAY_WINDOW,
        )

        models, optimizers, init_param_vecs = [], [], []
        forget_auc_init, forget_loss_init = [], []
        for si in range(N_SEEDS):
            seed = SEED_BASE + si
            model = init_fit_mlp_v2(X_fit, y_fit, X_val, y_val, n_features, seed)
            optimizer = torch.optim.Adam(model.parameters(), lr=V2_LEARNING_RATE, weight_decay=V2_WEIGHT_DECAY)
            models.append(model); optimizers.append(optimizer)
            init_param_vecs.append(flat_params(model).clone())
            with torch.no_grad():
                p_val = torch.sigmoid(model(torch.tensor(X_val, dtype=torch.float32))).numpy()
            forget_auc_init.append(roc_auc_score(y_val, p_val) if len(np.unique(y_val)) > 1 else np.nan)
            forget_loss_init.append(float(nn.functional.binary_cross_entropy(
                torch.tensor(p_val), torch.tensor(y_val, dtype=torch.float32)).item()))

        oof_rows = []
        n_updates = 0
        for idx, d in enumerate(dates_year):
            if idx > 0:
                d_prev = dates_year[idx - 1]
                prev_rows = year_rows[year_rows["date"] == d_prev]
                replay_frame = pd.concat([prev_rows] + list(buffer), ignore_index=True)
                X_replay = scaler.transform(replay_frame[feature_cols])
                y_replay = replay_frame["label_binary"].to_numpy()
                for si in range(N_SEEDS):
                    one_incremental_epoch(models[si], optimizers[si], X_replay, y_replay, loss_fn)
                buffer.append(prev_rows)
                n_updates += 1

            rows_d = year_rows[year_rows["date"] == d]
            X_d = scaler.transform(rows_d[feature_cols])
            with torch.no_grad():
                seed_preds = [torch.sigmoid(models[si](torch.tensor(X_d, dtype=torch.float32))).numpy()
                              for si in range(N_SEEDS)]
            pred = np.mean(seed_preds, axis=0)
            out = rows_d[["date", "symbol", "next_week_open_to_close_return", "next_week_vti_return",
                           "label_binary"]].copy()
            out["predicted_proba"] = pred
            oof_rows.append(out)

        oof = pd.concat(oof_rows, ignore_index=True)
        with torch.no_grad():
            p_val_end = [torch.sigmoid(models[si](torch.tensor(X_val, dtype=torch.float32))).numpy()
                         for si in range(N_SEEDS)]
        auc_drops = []
        for si in range(N_SEEDS):
            auc_end = roc_auc_score(y_val, p_val_end[si]) if len(np.unique(y_val)) > 1 else np.nan
            drop = forget_auc_init[si] - auc_end if not np.isnan(forget_auc_init[si]) and not np.isnan(auc_end) else np.nan
            auc_drops.append(drop)

        y_oof = oof["label_binary"].to_numpy(); s_oof = oof["predicted_proba"].to_numpy()
        return oof, {
            "year": year, "n_weekly_updates": n_updates,
            "year_auc": roc_auc_score(y_oof, s_oof) if len(np.unique(y_oof)) > 1 else np.nan,
            "year_pr_auc": average_precision_score(y_oof, s_oof) if len(np.unique(y_oof)) > 1 else np.nan,
            "year_brier": brier_score_loss(y_oof, s_oof),
            "mean_auc_drop_vs_init": float(np.nanmean(auc_drops)), "max_auc_drop_vs_init": float(np.nanmax(auc_drops)),
        }

    print(f"[{time.time()-t0:.0f}s] R6_REPLAY_SENSITIVITY triggered for NN_v2 (REPLAY_WINDOW={REPLAY_WINDOW}). Running...")
    replay_rows = []
    oof_replay_by_year = {}
    for year in R6_TRADING_YEARS:
        oof_r, row_r = run_stateful_year_v2_replay(md, year, FULL_FEATURE_COLS, diag_store=[], forgetting_store=[])
        replay_rows.append(row_r); oof_replay_by_year[year] = oof_r
        print(f"[{time.time()-t0:.0f}s] replay NN_v2 year={year}: auc={row_r['year_auc']:.4f} "
              f"mean_auc_drop={row_r['mean_auc_drop_vs_init']:.4f}")

    oof_replay_all = pd.concat(oof_replay_by_year.values(), ignore_index=True)
    primary_v2_fold = r6_fold_metrics[r6_fold_metrics.model == "NN_v2"].set_index("year")
    replay_df = pd.DataFrame(replay_rows).set_index("year")
    comparison_rows = []
    for year in R6_TRADING_YEARS:
        comparison_rows.append({
            "year": year,
            "primary_year_auc": primary_v2_fold.loc[year, "year_auc"],
            "replay_year_auc": replay_df.loc[year, "year_auc"],
            "primary_mean_auc_drop": r6_forgetting[(r6_forgetting.model == "NN_v2") & (r6_forgetting.year == year)]["auc_drop"].mean(),
            "replay_mean_auc_drop": replay_df.loc[year, "mean_auc_drop_vs_init"],
            "replay_n_updates": replay_df.loc[year, "n_weekly_updates"],
        })
    r6_replay_sensitivity = pd.DataFrame(comparison_rows)
    r6_replay_sensitivity.to_csv(OUTPUTS_ROOT / "r6_replay_sensitivity.csv", index=False)

    oof_r_common = oof_replay_all[oof_replay_all["date"].dt.year.isin(COMMON_YEARS)]
    oof_p_common = oof_v2_all[oof_v2_all["date"].dt.year.isin(COMMON_YEARS)]
    replay_auc_common = roc_auc_score(oof_r_common.label_binary, oof_r_common.predicted_proba)
    primary_auc_common = roc_auc_score(oof_p_common.label_binary, oof_p_common.predicted_proba)
    print(f"[{time.time()-t0:.0f}s] R6_REPLAY_SENSITIVITY done. NN_v2 common-period pooled AUC: "
          f"primary(no replay)={primary_auc_common:.4f} vs replay={replay_auc_common:.4f}. "
          f"mean auc_drop: primary={r6_forgetting[r6_forgetting.model=='NN_v2']['auc_drop'].mean():.4f} "
          f"vs replay={replay_df['mean_auc_drop_vs_init'].mean():.4f}")
    summary["replay_sensitivity"] = {
        "replay_window_weeks": REPLAY_WINDOW,
        "nn_v2_primary_auc_common_period": float(primary_auc_common),
        "nn_v2_replay_auc_common_period": float(replay_auc_common),
        "nn_v2_primary_mean_auc_drop": float(r6_forgetting[r6_forgetting.model == "NN_v2"]["auc_drop"].mean()),
        "nn_v2_replay_mean_auc_drop": float(replay_df["mean_auc_drop_vs_init"].mean()),
    }
    with open(OUTPUTS_ROOT / "r6_run_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
else:
    print(f"[{time.time()-t0:.0f}s] Replay sensitivity NOT triggered (forgetting classification: "
          f"{forgetting_overall}).")

print(f"[{time.time()-t0:.0f}s] ALL DONE.")
print(json.dumps(summary, indent=2))

[114s] R6_REPLAY_SENSITIVITY triggered for NN_v2 (REPLAY_WINDOW=12). Running...


[114s] replay NN_v2 year=2007: auc=0.6102 mean_auc_drop=-0.0338


[115s] replay NN_v2 year=2008: auc=0.5304 mean_auc_drop=0.0015


[115s] replay NN_v2 year=2009: auc=0.4268 mean_auc_drop=0.0074


[115s] replay NN_v2 year=2010: auc=0.5420 mean_auc_drop=-0.0051


...

[120s] replay NN_v2 year=2022: auc=0.5087 mean_auc_drop=-0.0006


[120s] replay NN_v2 year=2023: auc=0.5156 mean_auc_drop=-0.0610


[120s] replay NN_v2 year=2024: auc=0.4589 mean_auc_drop=-0.0303


[121s] replay NN_v2 year=2025: auc=0.5543 mean_auc_drop=-0.0570


[121s] replay NN_v2 year=2026: auc=0.4944 mean_auc_drop=-0.0028
[121s] R6_REPLAY_SENSITIVITY done. NN_v2 common-period pooled AUC: primary(no replay)=0.5507 vs replay=0.5552. mean auc_drop: primary=-0.0010 vs replay=-0.0185
[121s] ALL DONE.
{
  "primary_feature_set": "A+B+C+D+E+F+G+H+I (full engineered set, 34 features) -- precedent from R3/R4",
  "r6_trading_years": [
    2007,
    2026
  ],
  "common_r1_comparison_years": [
    2008,
    2026
  ],
  "nn_v1_r6_pooled_auc_full_period": 0.5508415216841229,
  "nn_v1_r6_pooled_auc_common_period": 0.549099060557884,
  "nn_v2_r6_pooled_auc_full_period": 0.5528363275049839,
  "nn_v2_r6_pooled_auc_common_period": 0.5507479115479115,
  "r1_baseline_nn_v1_auc_full_feature_set": 0.5693678469913763,
  "r1_baseline_nn_v2_auc_full_feature_set": 0.5708322204557499,
  "max_auc_drop_forgetting": 0.0579230393183881,
  "forgetting_classification_overall": "material",
  "replay_sensitivity_triggered": true,
  "runtime_seconds": 113.73952794075012,
  "rep